In [35]:
import os
import re
import numpy as np

pattern = r'Throughput.*?Decode:\s*([0-9]+(?:\.[0-9]+)?)'
pattern2 = r"Average reuse rate:\s*([0-9]*\.?[0-9]+)"
pattern3 = r"sum_swap_bw:\s*([\d.]+)\s*MB/s.*?sum_flush_bw:\s*([\d.]+)\s*MB/s"

def extract(exp_path):
	log_dict = {}
	for seed_dir in os.listdir(exp_path):
		seed_path = os.path.join(exp_path, seed_dir)
		for log_file in os.listdir(seed_path):
			if log_file.endswith('.log'):
				log_path = os.path.join(seed_path, log_file)
				with open(log_path, 'r') as f:
					text = f.read()
				match = re.search(pattern, text)
				if match:
					throughput = match.group(1)
					match = re.search(pattern2, text)
					if match:
						rate = float(match.group(1))
						match = re.search(pattern3, text, re.S)
						if match:
							swap_bw, flush_bw = match.groups()
							swap_bw = float(swap_bw)
							flush_bw = float(flush_bw)
							log_name = os.path.splitext(log_file)[0]
							log_name = log_name.replace('100_L4_none_clear_1_', '').replace("_none_concat", "").replace('_flash', '')
							if log_name not in log_dict:
								log_dict[log_name] = []
							log_dict[log_name].append((seed_dir, throughput, rate, swap_bw, flush_bw))
						else:
							print(f"{seed_dir}, Log: {log_file}, No swap/flush bandwidth found")
							continue
					else:
						print(f"{seed_dir}, Log: {log_file}, No reuse rate found")
						continue
				else:
					print(f"{seed_dir}, Log: {log_file}, No throughput found")
					continue
	return log_dict


def print_log_dict(log_dict):
	# sort the log_dict by key
	log_dict = dict(sorted(log_dict.items()))
	for key, values in log_dict.items():
		print(f"Log: {key}")
		tp_values = []
		rates = []
		flush_bws = []
		swap_bws = []
		for seed_dir, tp, rate, swap_bw, flush_bw in values:
			# if flush_bw < 20:
				# print(f"Skipping {seed_dir} due to low flush bandwidth: {flush_bw} MB/s")
				# continue
			tp_values.append(float(tp))
			rates.append(rate * 100)
			flush_bws.append(flush_bw)
			swap_bws.append(swap_bw)
		if not tp_values:
			print(f"No valid data for {key}, skipping...")
			continue
		tp_values = np.array(tp_values)
		rates = np.array(rates)
		flush_bws = np.array(flush_bws)
		swap_bws = np.array(swap_bws)
		print("Throughput Values:", tp_values)
		print("Reuse Rate Values:", rates)
		print("Swap Bandwidth Values:", swap_bws)
		print("Flush Bandwidth Values:", flush_bws)
		print(f"Average Throughput: {np.mean(tp_values):.2f} tokens/sec")
		print(f"Min Throughput: {np.min(tp_values):.2f} tokens/sec")
		print(f"Max Throughput: {np.max(tp_values):.2f} tokens/sec")
		print(f"Std Dev Throughput: {np.std(tp_values):.2f} tokens/sec")
		print(f"Average Reuse Rate: {np.mean(rates):.2f}%")
		print(f"Min Reuse Rate: {np.min(rates):.2f}%")
		print(f"Max Reuse Rate: {np.max(rates):.2f}%")
		print(f"Std Dev Reuse Rate: {np.std(rates):.2f}%")
		print(f"Average Swap Bandwidth: {np.mean(swap_bws):.2f} MB/s")
		print(f"Average Flush Bandwidth: {np.mean(flush_bws):.2f} MB/s")
		print(f"Total Runs: {len(tp_values)}")

		print("\n")	
		print("\n")	
	
		# min_tp = np.min(tp_values)
		# max_tp = np.max(tp_values)
		# mean_tp = np.mean(tp_values)
		# median_tp = np.median(tp_values)
		# std_tp = np.std(tp_values)
		# min_rate = np.min(rates)
		# max_rate = np.max(rates)
		# mean_rate = np.mean(rates)
		# median_rate = np.median(rates)
		# std_rate = np.std(rates)
		# print(sorted(rates))
		# print('len(tp_values):', len(tp_values))
		# print(f"Min: {min_tp:.2f}, Max: {max_tp:.2f}, Mean: {mean_tp:.2f}, Std: {std_tp:.2f}, Median: {median_tp:.2f}")
		# print(sorted(tp_values))
		# print(f"Reuse Rate - Min: {min_rate:.2f}, Max: {max_rate:.2f}, Mean: {mean_rate:.2f}, Std: {std_rate:.2f}, Median: {median_rate:.2f}")
		# print("\n")


log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/nvme/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg1/ru400/lr_proj_mh')
# log_dict_noreuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/nvme/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg4/ru0/lr_proj_mh')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/emmc/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg8/ru400/lr_proj_mh')
# log_dict_noreuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/emmc/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg8/ru0/lr_proj_mh')

# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed_MSQue/log/orin-agx_MAXN_INF/nvme/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg4/ru400/lr_proj_mh')
# log_dict_noreuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed_MSQue/log/orin-agx_MAXN_INF/nvme/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg4/ru0/lr_proj_mh')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed_MSQue/log/orin-agx_MAXN_INF/emmc/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg8/ru400/lr_proj_mh')
# log_dict_noreuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed_MSQue/log/orin-agx_MAXN_INF/emmc/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg8/ru0/lr_proj_mh')

# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/emmc/Llama-3.2-3B-Instruct/ra0/eng0/paged1/tg8/ru0/lr_proj_mh')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/nvme/Llama-3.2-3B-Instruct/ra0/eng0/paged1/tg4/ru0/lr_proj_mh')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/nvme/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg4/ru400/lr_proj_mh')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/nvme/Qwen3-14B/ra0/eng0/paged1/tg4/ru0/lr_proj_mh')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/emmc/Qwen3-14B/ra0/eng0/paged1/tg8/ru400/lr_proj_mh')
# log_dict_noreuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/emmc/Llama-3.2-3B-Instruct/ra0/eng0/paged1/tg8/ru0/lr_proj_mh')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/emmc/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg1/ru400/base')
# log_dict_noreuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/emmc/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg1/ru0/base')
# log_dict_reuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/nvme/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg1/ru400/base')
# log_dict_noreuse = extract('/home/schz/ext_disk/kvswap_exp_0508_seed/log/orin-agx_MAXN_INF/nvme/Llama-3.1-8B-Instruct/ra0/eng0/paged1/tg1/ru0/base')

print("Reuse Rate Experiment Results:")
print("====================================")
print_log_dict(log_dict_reuse)
print("\n")
print("\n")

# print("No Reuse Rate Experiment Results:")
# print("====================================")
# print_log_dict(log_dict_noreuse)
# print("\n")
# print("\n")



seed54810, Log: 8-24476-100_L4_none_clear_1_400_0-curr-emb_none_concat_0_p1_flash.log, No throughput found
seed79017, Log: 8-24476-100_L4_none_clear_1_400_0-curr-emb_none_concat_0_p1_flash.log, No throughput found
Reuse Rate Experiment Results:
Log: 8-24476-400_0-curr-emb_0_p1
Throughput Values: [29.44 29.12 28.35 29.52 28.92 28.86 28.44 27.63]
Reuse Rate Values: [80. 76. 74. 78. 76. 75. 74. 76.]
Swap Bandwidth Values: [4685.2 2732.3 2707.6 2946.9 2761.8 2734.3 2662.  2742. ]
Flush Bandwidth Values: [11.5 10.3  8.6  9.7 10.   8.8  9.2  8.5]
Average Throughput: 28.79 tokens/sec
Min Throughput: 27.63 tokens/sec
Max Throughput: 29.52 tokens/sec
Std Dev Throughput: 0.59 tokens/sec
Average Reuse Rate: 76.12%
Min Reuse Rate: 74.00%
Max Reuse Rate: 80.00%
Std Dev Reuse Rate: 1.90%
Average Swap Bandwidth: 2996.51 MB/s
Average Flush Bandwidth: 9.57 MB/s
Total Runs: 8








